# Market Dollar Bars

Build fixed $1,000,000-threshold dollar bars from the observed AAPL trades for the downstream research workflow. The existing feature Parquet is reused when available.

## Process the Data

- **Purpose.** Aggregate observed AAPL trades into activity-based dollar bars.
- **Key settings.** `dollar_threshold=$1,000,000` of accumulated traded notional per bar.
- **Data & decision.** Close bars at the fixed threshold and reuse the cached Parquet artifact when present.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from src.data_preprocessing.market_structured_bars import (
    get_dollar_bars,
    save_structured_bar_result,
)

PROJECT_ROOT = Path.cwd().resolve().parents[1]
market_path = PROJECT_ROOT / "data/research_data/market/data/aapl_2025-01-01_2025-12-31.parquet"
feature_dir = PROJECT_ROOT / "data/research_data/market/features"
period = "2025-01-01_2025-12-31"
dollar_bar_path = feature_dir / f"aapl_dollar_bar_{period}.parquet"
if not dollar_bar_path.is_file():
    observations = pd.read_parquet(market_path)
    result = get_dollar_bars(observations, threshold=1_000_000.0)
    save_structured_bar_result(result, dollar_bar_path)
dollar_bars = pd.read_parquet(dollar_bar_path)
dollar_bar_path

PosixPath('/Users/kwonjunhyuk9/Documents/financial-machine-learning/data/research_data/market/features/aapl_dollar_bar_2025-01-01_2025-12-31.parquet')

## Take a Quick Look at the Data Structure

- **Purpose.** Inspect the bar schema, AAPL-only coverage, and OHLCV and notional ranges.
- **Key settings.** No analytical parameters; read-only inspection.
- **Data & decision.** The checks do not rebuild bars or modify the cached dollar-bar artifact.

In [2]:
dollar_bars.head()

,end,start,symbol,open,high,low,close,volume,dollar_value,ticks,buy_volume,sell_volume
0,2025-01-02 14:30:02.119118+00:00,2025-01-02 13:45:58.902948+00:00,AAPL,250.00,250.000,248.61,248.650,4173.0,1038511.620,65,2174.0,1999.0
1,2025-01-02 14:30:04.394582+00:00,2025-01-02 14:30:02.129080+00:00,AAPL,248.59,248.715,248.56,248.610,4031.0,1002299.490,64,1149.0,2882.0
2,2025-01-02 14:30:09.443442+00:00,2025-01-02 14:30:04.394604+00:00,AAPL,248.61,248.840,248.53,248.805,4066.0,1010971.660,51,2464.0,1602.0
3,2025-01-02 14:30:12.024260+00:00,2025-01-02 14:30:09.575397+00:00,AAPL,248.85,248.850,248.60,248.680,4088.0,1017030.510,29,1431.0,2657.0
4,2025-01-02 14:30:40.602912+00:00,2025-01-02 14:30:12.024272+00:00,AAPL,248.68,248.690,248.41,248.525,4119.0,1023735.835,56,2742.0,1377.0


In [3]:
dollar_bars.info()

<class 'pandas.DataFrame'>
RangeIndex: 80213 entries, 0 to 80212
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype              
---  ------        --------------  -----              
 0   end           80213 non-null  datetime64[us, UTC]
 1   start         80213 non-null  datetime64[us, UTC]
 2   symbol        80213 non-null  str                
 3   open          80213 non-null  float64            
 4   high          80213 non-null  float64            
 5   low           80213 non-null  float64            
 6   close         80213 non-null  float64            
 7   volume        80213 non-null  float64            
 8   dollar_value  80213 non-null  float64            
 9   ticks         80213 non-null  int64              
 10  buy_volume    80213 non-null  float64            
 11  sell_volume   80213 non-null  float64            
dtypes: datetime64[us, UTC](2), float64(8), int64(1), str(1)
memory usage: 7.6 MB


In [4]:
dollar_bars["symbol"].value_counts(dropna=False)

symbol
AAPL    80213
Name: count, dtype: int64

In [5]:
dollar_bars.describe()

,open,high,low,close,volume,dollar_value,ticks,buy_volume,sell_volume
count,80213.000000,80213.000000,80213.000000,80213.000000,80213.000000,8.021300e+04,80213.000000,80213.000000,80213.000000
mean,233.333252,233.451244,233.213955,233.333483,4486.747747,1.031835e+06,55.658160,2283.570057,2203.177689
std,28.195120,28.176175,28.214072,28.195407,940.612032,1.822302e+05,17.474442,936.919463,971.470907
min,169.480000,169.560000,168.620000,169.490000,3471.000000,1.000000e+06,1.000000,0.000000,0.000000
25%,209.625000,209.755000,209.495000,209.620000,3957.000000,1.005712e+06,45.000000,1771.000000,1688.000000
50%,231.720000,231.860000,231.580000,231.710000,4399.000000,1.012638e+06,55.000000,2230.000000,2150.000000
75%,257.210000,257.335000,257.100000,257.220000,4886.000000,1.020913e+06,66.000000,2717.000000,2642.000000
max,288.540000,288.610000,287.980000,288.550000,78973.000000,2.021433e+07,395.000000,51959.000000,78813.000000


In [6]:
dollar_bars.hist(bins=50, figsize=(14, 10))
plt.tight_layout()
plt.show()

/var/folders/1z/bcvql7210c77v6rjkswpzsyr0000gn/T/ipykernel_82277/2908202076.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
